# Spotify Tracks - Comparative Clustering Analysis

This notebook performs a comprehensive comparative study of **16 clustering algorithms** on the Spotify Tracks preprocessed subset. We evaluate partitioning, probabilistic, hierarchical, density-based, grid-based, and extra algorithms using internal validation metrics.

In [ ]:
import os
import sys
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.base import BaseEstimator, ClusterMixin
from sklearn.decomposition import PCA
from sklearn.manifold import MDS
from sklearn.decomposition import FastICA
from sklearn.cluster import KMeans, BisectingKMeans, DBSCAN, OPTICS, SpectralClustering, Birch
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, silhouette_samples, calinski_harabasz_score, davies_bouldin_score
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

# Set style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

## 1. Load Preprocessed Data

We load the preprocessed and sampled tracks dataset, then standardize features again to be absolutely sure of normalization.

In [ ]:
data_path = "output/spotify_sampled_preprocessed.csv"
if not os.path.exists(data_path):
    data_path = "../output/spotify_sampled_preprocessed.csv"  # fallback if running from notebooks/

if not os.path.exists(data_path):
    print("Error: Preprocessed dataset not found. Please run the eda_and_preprocessing notebook first.")
else:
    df_sampled = pd.read_csv(data_path)
    features = [
        'popularity', 'duration_ms', 'danceability', 'energy', 'key', 
        'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 
        'liveness', 'valence', 'tempo', 'time_signature', 'explicit'
    ]
    df_sampled['explicit'] = df_sampled['explicit'].astype(float)
    X = df_sampled[features].values
    X_scaled = StandardScaler().fit_transform(X)
    print("Sample loaded. Feature matrix shape:", X_scaled.shape)

## 2. Custom Algorithms Implementations

We define our custom classes for **STING** and **DENCLUE** written from scratch in NumPy.

In [ ]:
class CustomDENCLUE(BaseEstimator, ClusterMixin):
    def __init__(self, h=1.5, eps=1e-4, min_density=0.0001, max_iter=50):
        self.h = h
        self.eps = eps
        self.min_density = min_density
        self.max_iter = max_iter

    def fit(self, X, y=None):
        self.X_fit_ = np.asarray(X)
        n_samples, n_features = self.X_fit_.shape
        attractors = np.zeros_like(self.X_fit_)
        densities = np.zeros(n_samples)
        
        for i in range(n_samples):
            point = self.X_fit_[i].copy()
            for _ in range(self.max_iter):
                diff = self.X_fit_ - point
                dist_sq = np.sum(diff**2, axis=1)
                weights = np.exp(-dist_sq / (2 * (self.h ** 2)))
                sum_weights = np.sum(weights)
                if sum_weights == 0: break
                new_point = np.sum(self.X_fit_ * weights[:, np.newaxis], axis=0) / sum_weights
                if np.sqrt(np.sum((new_point - point)**2)) < self.eps:
                    point = new_point
                    break
                point = new_point
            attractors[i] = point
            diff_attr = self.X_fit_ - point
            dist_sq_attr = np.sum(diff_attr**2, axis=1)
            densities[i] = np.mean(np.exp(-dist_sq_attr / (2 * (self.h ** 2))))
            
        labels = -np.ones(n_samples, dtype=int)
        unique_attractors = []
        cluster_id = 0
        for i in range(n_samples):
            if densities[i] < self.min_density:
                labels[i] = -1
                continue
            found = False
            for c_id, attr in enumerate(unique_attractors):
                if np.sqrt(np.sum((attractors[i] - attr)**2)) < (self.h / 2.0):
                    labels[i] = c_id
                    found = True
                    break
            if not found:
                unique_attractors.append(attractors[i])
                labels[i] = cluster_id
                cluster_id += 1
        self.labels_ = labels
        return self


class CustomSTING(BaseEstimator, ClusterMixin):
    def __init__(self, grid_size=25, min_samples=5):
        self.grid_size = grid_size
        self.min_samples = min_samples

    def fit(self, X, y=None):
        X = np.asarray(X)
        n_samples, n_features = X.shape
        pca = PCA(n_components=2, random_state=42)
        X_2d = pca.fit_transform(X)
        
        x_min, x_max = X_2d[:, 0].min() - 1e-6, X_2d[:, 0].max() + 1e-6
        y_min, y_max = X_2d[:, 1].min() - 1e-6, X_2d[:, 1].max() + 1e-6
        
        x_bins = np.linspace(x_min, x_max, self.grid_size + 1)
        y_bins = np.linspace(y_min, y_max, self.grid_size + 1)
        
        x_indices = np.digitize(X_2d[:, 0], x_bins) - 1
        y_indices = np.digitize(X_2d[:, 1], y_bins) - 1
        
        grid = np.zeros((self.grid_size, self.grid_size), dtype=int)
        cell_points = {}
        for i in range(n_samples):
            gx, gy = x_indices[i], y_indices[i]
            grid[gx, gy] += 1
            if (gx, gy) not in cell_points:
                cell_points[(gx, gy)] = []
            cell_points[(gx, gy)].append(i)
            
        active_cells = grid >= self.min_samples
        labels = -np.ones(n_samples, dtype=int)
        visited = np.zeros((self.grid_size, self.grid_size), dtype=bool)
        cluster_id = 0
        
        for gx in range(self.grid_size):
            for gy in range(self.grid_size):
                if active_cells[gx, gy] and not visited[gx, gy]:
                    queue = [(gx, gy)]
                    visited[gx, gy] = True
                    component_points = []
                    while queue:
                        cx, cy = queue.pop(0)
                        if (cx, cy) in cell_points:
                            component_points.extend(cell_points[(cx, cy)])
                        for dx in [-1, 0, 1]:
                            for dy in [-1, 0, 1]:
                                if dx == 0 and dy == 0: continue
                                nx, ny = cx + dx, cy + dy
                                if 0 <= nx < self.grid_size and 0 <= ny < self.grid_size:
                                    if active_cells[nx, ny] and not visited[nx, ny]:
                                        visited[nx, ny] = True
                                        queue.append((nx, ny))
                    if component_points:
                        for p_idx in component_points:
                            labels[p_idx] = cluster_id
                        cluster_id += 1
        self.labels_ = labels
        return self

## 3. Elbow Analysis

We perform Elbow Analysis for KMeans (Inertia SSE) and Gaussian Mixture (AIC/BIC) to choose cluster size $K$ values.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# KMeans Elbow
inertias = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=5).fit(X_scaled)
    inertias.append(km.inertia_)
axes[0].plot(range(2, 11), inertias, 'bo-')
axes[0].set_xlabel('K')
axes[0].set_ylabel('Inertia (SSE)')
axes[0].set_title('K-Means Elbow Plot')

# GMM AIC/BIC
aics, bics = [], []
for k in range(2, 11):
    gmm = GaussianMixture(n_components=k, random_state=42, n_init=1).fit(X_scaled)
    aics.append(gmm.aic(X_scaled))
    bics.append(gmm.bic(X_scaled))
axes[1].plot(range(2, 11), aics, 'ro-', label='AIC')
axes[1].plot(range(2, 11), bics, 'go-', label='BIC')
axes[1].set_xlabel('K')
axes[1].set_ylabel('Information Criteria')
axes[1].set_title('GMM AIC/BIC')
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. Run Clustering and Evaluate Partitions

We run 15 algorithms (each with 2 partitions) and evaluate their internal validation metrics.

In [ ]:
partitions_results = []

def evaluate_partition(name, part_idx, labels):
    unique_labels = np.unique(labels)
    n_clusters = len(unique_labels) - (1 if -1 in unique_labels else 0)
    if n_clusters < 2:
        return None
    mask = labels != -1
    if np.sum(mask) < 2 or len(np.unique(labels[mask])) < 2:
        return None
        
    sil = silhouette_score(X_scaled[mask], labels[mask])
    sil_log = np.log(sil + 1.0)
    sil_sqrt = np.sign(sil) * np.sqrt(abs(sil))
    ch = calinski_harabasz_score(X_scaled[mask], labels[mask])
    db = davies_bouldin_score(X_scaled[mask], labels[mask])
    sample_sils = silhouette_samples(X_scaled, np.where(labels == -1, 9999, labels))
    
    return {
        'Algorithm': name, 'Partition': part_idx, 'Num_Clusters': n_clusters,
        'Silhouette': sil, 'Log_Silhouette': sil_log, 'Sqrt_Silhouette': sil_sqrt,
        'Calinski_Harabasz': ch, 'Davies_Bouldin': db, 'Labels': labels.copy(),
        'Sample_Silhouettes': sample_sils
    }

# 1. Partitioning
partitions_results.append(evaluate_partition("KMeans", 1, KMeans(n_clusters=3, random_state=42, n_init=5).fit(X_scaled).labels_))
partitions_results.append(evaluate_partition("KMeans", 2, KMeans(n_clusters=5, random_state=42, n_init=5).fit(X_scaled).labels_))
partitions_results.append(evaluate_partition("BisectingKMeans", 1, BisectingKMeans(n_clusters=3, random_state=42, n_init=5).fit(X_scaled).labels_))
partitions_results.append(evaluate_partition("BisectingKMeans", 2, BisectingKMeans(n_clusters=5, random_state=42, n_init=5).fit(X_scaled).labels_))

# Try KMedoids
try:
    from sklearn_extra.cluster import KMedoids
    partitions_results.append(evaluate_partition("KMedoids", 1, KMedoids(n_clusters=3, random_state=42).fit(X_scaled).labels_))
    partitions_results.append(evaluate_partition("KMedoids", 2, KMedoids(n_clusters=5, random_state=42).fit(X_scaled).labels_))
except ImportError:
    print("KMedoids not available. Skipping.")

# 2. Probabilistic
partitions_results.append(evaluate_partition("GaussianMixture", 1, GaussianMixture(n_components=3, random_state=42, n_init=2).fit(X_scaled).predict(X_scaled)))
partitions_results.append(evaluate_partition("GaussianMixture", 2, GaussianMixture(n_components=5, random_state=42, n_init=2).fit(X_scaled).predict(X_scaled)))

# 3. Hierarchical Linkages
for method, name in zip(['ward', 'complete', 'single', 'average', 'centroid'], ["Ward", "CompleteLinkage", "SimpleLinkage", "AverageLinkage", "CentroidLinkage"]):
    Z = linkage(X_scaled, method=method, metric='euclidean')
    partitions_results.append(evaluate_partition(name, 1, fcluster(Z, 3, criterion='maxclust') - 1))
    partitions_results.append(evaluate_partition(name, 2, fcluster(Z, 5, criterion='maxclust') - 1))

# 4. Density-Based
partitions_results.append(evaluate_partition("DBSCAN", 1, DBSCAN(eps=1.5, min_samples=5).fit(X_scaled).labels_))
partitions_results.append(evaluate_partition("DBSCAN", 2, DBSCAN(eps=2.5, min_samples=5).fit(X_scaled).labels_))
partitions_results.append(evaluate_partition("OPTICS", 1, OPTICS(min_samples=10, xi=0.05).fit(X_scaled).labels_))

try:
    from sklearn.cluster import HDBSCAN
    partitions_results.append(evaluate_partition("HDBSCAN", 1, HDBSCAN(min_cluster_size=15).fit(X_scaled).labels_))
    partitions_results.append(evaluate_partition("HDBSCAN", 2, HDBSCAN(min_cluster_size=30).fit(X_scaled).labels_))
except ImportError:
    print("HDBSCAN not available in this sklearn version. Skipping.")

# 5. Grid-Based
partitions_results.append(evaluate_partition("STING", 1, CustomSTING(grid_size=25, min_samples=5).fit(X_scaled).labels_))
partitions_results.append(evaluate_partition("STING", 2, CustomSTING(grid_size=30, min_samples=4).fit(X_scaled).labels_))
partitions_results.append(evaluate_partition("DENCLUE", 1, CustomDENCLUE(h=1.5, min_density=0.0001).fit(X_scaled).labels_))
partitions_results.append(evaluate_partition("DENCLUE", 2, CustomDENCLUE(h=2.0, min_density=0.0001).fit(X_scaled).labels_))

# 6. Extras (Spectral, BIRCH)
partitions_results.append(evaluate_partition("SpectralClustering", 1, SpectralClustering(n_clusters=3, random_state=42, assign_labels='discretize', n_init=2).fit(X_scaled).labels_))
partitions_results.append(evaluate_partition("SpectralClustering", 2, SpectralClustering(n_clusters=5, random_state=42, assign_labels='discretize', n_init=2).fit(X_scaled).labels_))
partitions_results.append(evaluate_partition("BIRCH", 1, Birch(n_clusters=3).fit(X_scaled).labels_))
partitions_results.append(evaluate_partition("BIRCH", 2, Birch(n_clusters=5).fit(X_scaled).labels_))

# Clean results list
partitions_results = [r for r in partitions_results if r is not None]
print(f"Completed. Evaluated {len(partitions_results)} partitions.")

## 5. Ranking and Comparison

We calculate a synthetic score ($0.5 \cdot Silhouette + 0.25 \cdot Calinski + 0.25 \cdot Davies\_Invers$) to rank all partitions.

In [ ]:
res_df = pd.DataFrame([{
    'Algorithm': r['Algorithm'], 'Partition': r['Partition'], 'Num_Clusters': r['Num_Clusters'],
    'Silhouette': r['Silhouette'], 'Log_Silhouette': r['Log_Silhouette'], 'Sqrt_Silhouette': r['Sqrt_Silhouette'],
    'Calinski_Harabasz': r['Calinski_Harabasz'], 'Davies_Bouldin': r['Davies_Bouldin']
} for r in partitions_results])

sil_min, sil_max = res_df['Silhouette'].min(), res_df['Silhouette'].max()
res_df['Scaled_Silhouette'] = (res_df['Silhouette'] - sil_min) / (sil_max - sil_min + 1e-9)

ch_min, ch_max = res_df['Calinski_Harabasz'].min(), res_df['Calinski_Harabasz'].max()
res_df['Scaled_Calinski'] = (res_df['Calinski_Harabasz'] - ch_min) / (ch_max - ch_min + 1e-9)

db_min, db_max = res_df['Davies_Bouldin'].min(), res_df['Davies_Bouldin'].max()
res_df['Scaled_Davies'] = 1.0 - (res_df['Davies_Bouldin'] - db_min) / (db_max - db_min + 1e-9)

res_df['Synthetic_Score'] = 0.5 * res_df['Scaled_Silhouette'] + 0.25 * res_df['Scaled_Calinski'] + 0.25 * res_df['Scaled_Davies']
res_df = res_df.sort_values(by='Synthetic_Score', ascending=False).reset_index(drop=True)

print("--- Top 10 Ranked Partitions ---")
display(res_df.head(10))

## 6. Projections & Visualization (Best Partition)

Let's extract and project the best partition using PCA, ICA, and MDS to see the 2D cluster layout.

In [ ]:
best_row = res_df.iloc[0]
best_algo = best_row['Algorithm']
best_part = best_row['Partition']
print(f"Best Partition: {best_algo} (Partition {best_part}) with {int(best_row['Num_Clusters'])} clusters")

best_data = next(r for r in partitions_results if r['Algorithm'] == best_algo and r['Partition'] == best_part)
labels = best_data['Labels']
n_clusters = best_data['Num_Clusters']

# Calculate 2D coordinates
pca_2d = PCA(n_components=2, random_state=42).fit_transform(X_scaled)
ica_2d = FastICA(n_components=2, random_state=42).fit_transform(X_scaled)
mds_2d = MDS(n_components=2, max_iter=50, n_init=1, random_state=42).fit_transform(X_scaled)

# Setup color maps
colors = plt.cm.get_cmap('tab10', max(10, n_clusters))
scatter_colors = [(0.7, 0.7, 0.7, 0.5) if l == -1 else colors(l % 10) for l in labels]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(pca_2d[:, 0], pca_2d[:, 1], c=scatter_colors, s=15, edgecolors='none')
axes[0].set_title('PCA Projection')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')

axes[1].scatter(ica_2d[:, 0], ica_2d[:, 1], c=scatter_colors, s=15, edgecolors='none')
axes[1].set_title('ICA Projection')
axes[1].set_xlabel('IC1')
axes[1].set_ylabel('IC2')

axes[2].scatter(mds_2d[:, 0], mds_2d[:, 1], c=scatter_colors, s=15, edgecolors='none')
axes[2].set_title('MDS Projection')
axes[2].set_xlabel('Dim 1')
axes[2].set_ylabel('Dim 2')

plt.suptitle(f'{best_algo} (Partition {best_part}) - 2D Cluster Visualizations', fontsize=14)
plt.tight_layout()
plt.show()

## 7. Silhouette Plot

Let's visualize the instance-level Silhouette scores for the best partition.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
y_lower = 10
sample_sils = best_data['Sample_Silhouettes']
avg_sil = best_data['Silhouette']

mask = labels != -1
cluster_labels = np.unique(labels[mask])

for i, c in enumerate(cluster_labels):
    ith_cluster_sil = sample_sils[labels == c]
    ith_cluster_sil.sort()
    size_i = ith_cluster_sil.shape[0]
    y_upper = y_lower + size_i
    
    color = colors(i % 10)
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, ith_cluster_sil,
                     facecolor=color, edgecolor=color, alpha=0.7)
    ax.text(-0.05, y_lower + 0.5 * size_i, str(c))
    y_lower = y_upper + 10

ax.set_title(f"Silhouette Plot: {best_algo} - Partition {best_part}")
ax.set_xlabel("Silhouette coefficient")
ax.set_ylabel("Cluster label")
ax.axvline(x=avg_sil, color="red", linestyle="--", label=f"Average S = {avg_sil:.3f}")
ax.set_yticks([])
ax.set_xlim([-0.2, 1.0])
ax.legend()
plt.show()

## 8. Cluster Profiling

Let's look at key musical attributes across the clusters to identify what makes each segment distinct.

In [ ]:
df_sampled['cluster_label'] = labels
df_profile = df_sampled[df_sampled['cluster_label'] != -1]
profile_features = ['danceability', 'energy', 'acousticness', 'valence', 'tempo', 'popularity']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.ravel()

for i, feat in enumerate(profile_features):
    sns.boxplot(x='cluster_label', y=feat, data=df_profile, ax=axes[i], palette='tab10')
    axes[i].set_title(f'{feat.capitalize()} distribution')
    axes[i].set_xlabel('Cluster')
    axes[i].set_ylabel(feat)

plt.suptitle('Audio Characteristics Profile per Cluster', fontsize=16)
plt.tight_layout()
plt.show()

### Genre Distribution

Let's see what are the top 5 genres in each cluster of our best partition.

In [ ]:
genre_dist = df_profile.groupby(['cluster_label', 'track_genre']).size().unstack(fill_value=0)

for c in genre_dist.index:
    top_genres = genre_dist.loc[c].sort_values(ascending=False).head(5)
    print(f"\n--- Cluster {c} Top Genres ---")
    for g, count in top_genres.items():
        print(f"  * {g}: {count} tracks")